# Session 2 · Day 1 - Institutions and Coordinate-Quality Audit

The TESDA assessment-center list arrives as hand-typed rows: many accreditations per center, coordinates typed across 17 regions, and no shared institution ID to join on. This notebook loads the real published files, audits and repairs the coordinates with a `coord_quality` flag, checks whether each pin agrees with its recorded address, and links qualifications and programs to the institution master by name and address. *Modeling is Session 3.*

### What this notebook covers
1. Load the real accredited-assessment-center file and read its grain: one row is one accreditation
2. Audit coordinate quality and repair the reversible defects (swap, dropped decimal), with a `coord_quality` flag
3. Spot the flaw in trusting raw coordinates without the flag
4. Compare a pin against its recorded address through the cleaned PSGC master, and count centers by address against by pin
5. Link qualifications and programs to the institution master without a shared ID, and report the link rate
6. Play around: move the Philippine bounding box and watch the flag counts move

> **How to use this notebook**
> - ▶ **Run each cell in order** (`Shift+Enter`). Later cells depend on earlier ones.
> - ✏️ **Exercise cells** have `### START CODE HERE ###` … `### END CODE HERE ###`. Fill in code between those lines only.
> - ✅ After an exercise, run the **check cell**. It recomputes the reference and reports a pass or a fix to make.
> - 🎛️ **Play-around cells** are open to tweak: change a value, re-run, compare.
>
> _Real data loads over the network from a public bucket. No Google Drive mount and no local paths are needed._

### New to Python or pandas?

Anyone coming from another language can run `S2_commands_primer` first. It lays out the mental model in one page: a `DataFrame` is a table (a SQL table, an R `data.frame`, a spreadsheet), a `merge` is a `JOIN`, `NaN` is the missing value, and a chain of `.method().method()` reads left to right. The comments in the code cells point out each pandas idiom as it appears.

In [ ]:
# Run once per session (skip if already installed).
!pip -q install pandas pyarrow openpyxl

In [ ]:
import numpy as np
import pandas as pd
import re

BASE = "https://storage.googleapis.com/tesda-datasets/data/"
pd.set_option("display.max_columns", 30)
pd.set_option("display.max_colwidth", 60)

## The accredited-assessment-center file

Load the 2026 Accredited Competency Assessment Center list straight from the public bucket. Read every column as text, so nothing gets silently converted: a coordinate typed as `7108138` stays visible exactly as typed. The file also carries personal data (a center manager and a phone number). Drop both columns at the door, before anything is displayed or saved. That is the privacy rule in practice: keep only what the analysis needs.

In [ ]:
AC_URL = BASE + "2026%20Accredited%20Compentency%20Assessment%20Center%20based%20on%20July%2015%2C%202026%20database.csv"
ac = pd.read_csv(AC_URL, dtype=str)
ac = ac.apply(lambda s: s.str.strip())            # stray spaces, e.g. "TVET "
ac = ac.drop(columns=["Center Manager", "Phone"]) # personal data, never kept
print(f"{len(ac):,} rows x {ac.shape[1]} columns")
ac.head(3)

### One row is one accreditation

The most important question to ask of any table: what does one row mean? Here a row is one **accreditation**, one center accredited to assess one qualification. Counting rows counts accreditations, so a center that assesses ten qualifications is counted ten times. The longitude header is also misspelled `Longtitude` in this file, a small trap for any script that expects `Longitude`.

In [ ]:
ac["center_key"] = (ac["Assessment Center"].str.upper().str.replace(r"\s+", " ", regex=True)
                    + " | " + ac["City Name"].str.upper())
print(f"accreditations (rows):        {len(ac):,}")
print(f"distinct centers (name+city): {ac['center_key'].nunique():,}")
print(f"distinct qualifications:      {ac['Qualification'].nunique():,}")

**Expected output:** 12,871 accreditations across 2,727 distinct centers and 227 qualifications. One center carries many rows.

## Coordinate-quality audit and repair

Coordinates in this file were typed by hand across 17 regions, so defects are normal. Each row gets a status. Five patterns matter, and two of them can be repaired safely because the repair is reversible:

| Status | What happened | Repair |
|---|---|---|
| `ok` | a valid number inside the Philippines | keep |
| `swapped_fixed` | latitude and longitude typed in each other's column | swap back |
| `decimal_fixed` | the decimal point was dropped (`7108138` to `7.108138`) | restore it, only when one reading is unambiguous |
| `placeholder` | the same number twice, or tiny values like 1 or 2 | drop |
| `out_of_ph` | a real number, but outside the Philippines (a pin in London) | drop, send back for correction |
| `missing` | blank | drop |

A swap or a dropped decimal has one correct reading, so the repair is safe and recorded. A placeholder, an out-of-country pin, or a blank has no reversible fix, so those rows keep no coordinate and carry the flag instead. Downstream work reads the flag and never trusts a raw coordinate on its own.

In [ ]:
LAT_R, LON_R = (4.2, 21.5), (116.0, 127.0)   # a generous Philippine bounding box

def to_num(s):
    try:
        return float(str(s).replace(",", "").strip())
    except (TypeError, ValueError):
        return np.nan

def in_range(v, r):
    return v == v and r[0] <= v <= r[1]

def shift_decimal(s, r, min_digits):
    s = str(s).strip()
    if not re.fullmatch(r"-?\d+", s) or len(s.lstrip("-")) < min_digits:
        return None
    v = abs(int(s))
    hits = [v / 10**k for k in range(1, 12) if in_range(v / 10**k, r)]
    return hits[0] if len(hits) == 1 else None

def check_coords(lat_s, lon_s):
    la, lo = to_num(lat_s), to_num(lon_s)
    if la != la or lo != lo:                        return "missing", np.nan, np.nan
    if in_range(la, LAT_R) and in_range(lo, LON_R): return "ok", la, lo
    if in_range(lo, LAT_R) and in_range(la, LON_R): return "swapped_fixed", lo, la
    fla = la if in_range(la, LAT_R) else shift_decimal(lat_s, LAT_R, 7)
    flo = lo if in_range(lo, LON_R) else shift_decimal(lon_s, LON_R, 8)
    ambiguous = (shift_decimal(lat_s, LON_R, 8) is not None
                 and shift_decimal(lon_s, LAT_R, 7) is not None)
    if fla is not None and flo is not None and not ambiguous:
        return "decimal_fixed", fla, flo
    if la == lo or (abs(la) < 4 and abs(lo) < 4):   return "placeholder", np.nan, np.nan
    return "out_of_ph", np.nan, np.nan

res = [check_coords(a, b) for a, b in zip(ac["Latitude"], ac["Longtitude"])]
ac["coord_quality"] = [r[0] for r in res]
ac["lat"] = [r[1] for r in res]
ac["lon"] = [r[2] for r in res]
ac["coord_quality"].value_counts()

**Expected output** (accreditation rows by status):

```
ok               11328
swapped_fixed     1122
out_of_ph          268
decimal_fixed       77
missing             58
placeholder         18
```

Over 1,100 rows had latitude and longitude swapped, and 77 had a dropped decimal point. Both sets are now repaired and carry a status that says so.

Inspect a few rows of each non-`ok` status. The `lat`/`lon` columns hold the repaired value where a repair was possible, and a blank where none was.

In [ ]:
ex = (ac[ac["coord_quality"] != "ok"]
      .groupby("coord_quality", group_keys=False).head(3)
      [["Assessment Center", "City Name", "Latitude", "Longtitude", "coord_quality", "lat", "lon"]])
ex.sort_values("coord_quality")

## 🐞 Spot the flaw - trusting raw coordinates

A common shortcut: treat every row whose latitude and longitude parse as numbers as mappable, and skip the audit. Run the naive count below, then compare it with the flag-based count. Read the gap before moving on.

In [ ]:
# naive: "if latitude and longitude are numbers, the pin is usable."
naive_usable = (ac["Latitude"].map(to_num).notna() & ac["Longtitude"].map(to_num).notna()).sum()

# flag-based: a pin is usable only when the audit kept a coordinate.
flag_usable = ac["lat"].notna().sum()

print(f"naive 'usable' rows (any numeric lat/lon): {naive_usable:,}")
print(f"flag-based usable rows (audit kept a pin): {flag_usable:,}")
print(f"rows the naive count would map to the wrong place: {naive_usable - flag_usable:,}")

The naive count is larger. Every placeholder and every out-of-country pin is still a pair of numbers, so a numeric test accepts it. A pin in London parses perfectly and lands the center in the wrong hemisphere. The `coord_quality` flag is the fix: keep a coordinate only when the audit could validate or safely repair it, and carry the flag so later steps exclude or down-weight the rest.

## ✏️ Exercise - usable-pin rate for each region

A coverage map is only as honest as its coordinates. For **each region** (`Region Name`), compute the share of accreditation rows whose audit kept a usable pin, that is rows where `lat` is not null. Sort ascending so the worst-covered region comes first. Store the result in `usable_rate`.

**Hint:** build a boolean column with `ac["lat"].notna()`, then `groupby("Region Name")[...].mean()`.

In [ ]:
### START CODE HERE ###   (about 2 lines)
usable_rate = None    # per-region share of rows with a usable pin, sorted ascending
### END CODE HERE ###
print(usable_rate)

## ✅ Check the work
Run this. It recomputes the reference and compares it to `usable_rate`.

In [ ]:
try:
    ref = (ac.assign(has_pin=ac["lat"].notna())
             .groupby("Region Name")["has_pin"].mean().sort_values())
    assert usable_rate is not None, "usable_rate is None - fill in the code."
    assert isinstance(usable_rate, pd.Series), f"expected a pandas Series, got {type(usable_rate)}"
    assert set(usable_rate.index) == set(ref.index), "some regions are missing"
    assert np.allclose(usable_rate.sort_index().values, ref.sort_index().values), \
        "values do not match - recompute the share of usable pins per region"
    assert list(usable_rate.index) == list(ref.index), "sort ascending (worst-covered region first)"
    print("✅ Passed! Per-region usable-pin rate matches the reference.")
except AssertionError as e:
    print("❌ Not yet:", e)

### ❓ Question

The audit repaired two kinds of defect: swapped latitude/longitude and a dropped decimal point. How many accreditation rows were repaired in total, that is how many carry `coord_quality` equal to `swapped_fixed` or `decimal_fixed`?

In [ ]:
q_repaired = None   # <- answer here (an integer)

In [ ]:
_repaired = int(ac["coord_quality"].isin(["swapped_fixed", "decimal_fixed"]).sum())
if q_repaired == _repaired:
    print(f"✅ Correct! {_repaired} rows were repaired.")
else:
    print(f"❌ Not yet: expected {_repaired}, got {q_repaired}.")

## Pin against address, through the PSGC master

A second published file, `tesda_coordinates.parquet`, is the cleaned geocoded master: one row per institution per source, with checked coordinates and the PSGC codes of the area each pin falls in (`psgc_observed_*`). That gives two independent descriptions of where an institution is: the recorded `region`, and `psgc_observed_region`, the region its pin actually lands in. When the two disagree, one of them is wrong. No boundary shapefile is needed to compare pin against address.

In [ ]:
MASTER_URL = BASE + "tesda_coordinates.parquet"
master = pd.read_parquet(MASTER_URL)
master["source_record"] = np.where(master["city_municipality"].isna(), "assessment", "training")
print(f"{len(master):,} rows x {master.shape[1]} columns")
print(master["source_record"].value_counts().to_dict())
master[["tesda_inst_id", "name", "region", "coord_status", "psgc_observed_region"]].head(3)

### Which region does the pin land in?

`psgc_observed_region` is a PSGC code (`13` for NCR, `19` for BARMM), while `region` is a name. Learn the name-to-code map from the data itself: for each recorded region, take the most common observed code. A row whose observed code differs from its region's usual code is a pin that fell in another region. This is the same idea as a spatial-join mismatch, read off the pre-computed PSGC codes.

Counting then splits into two: **by address** uses the recorded region, **by pin** uses the observed code. The gap between the two is the coordinate-quality problem, measured.

In [ ]:
has_code = master[master["psgc_observed_region"].notna()].copy()
region_to_code = has_code.groupby("region")["psgc_observed_region"].agg(lambda s: s.mode().iloc[0])

has_code["expected_code"] = has_code["region"].map(region_to_code)
mismatch = int((has_code["psgc_observed_region"] != has_code["expected_code"]).sum())
print(f"pins that fall outside their recorded region: {mismatch:,} of {len(has_code):,}")

# "How many institutions in NCR?" by address vs by pin
ncr_code = region_to_code["NCR"]
print(f"NCR by address: {(master['region'] == 'NCR').sum():,}   "
      f"by pin: {(master['psgc_observed_region'] == ncr_code).sum():,}")

**Expected output:** 276 pins fall outside their recorded region. NCR holds 933 institutions by address against 917 by pin: the 16-row gap is centers whose recorded region and pin disagree.

## ✏️ Exercise - count a region by address and by pin

Count the institutions in `Region III` two ways. **By address**: rows whose `region` equals `"Region III"`. **By pin**: rows whose `psgc_observed_region` equals that region's code in `region_to_code`. Store the two integers in `r3_by_address` and `r3_by_pin`.

In [ ]:
### START CODE HERE ###   (about 2 lines)
r3_by_address = None    # rows whose region is "Region III"
r3_by_pin     = None    # rows whose psgc_observed_region equals Region III's code
### END CODE HERE ###
print("by address:", r3_by_address, " by pin:", r3_by_pin)

In [ ]:
try:
    _addr = int((master["region"] == "Region III").sum())
    _pin  = int((master["psgc_observed_region"] == region_to_code["Region III"]).sum())
    assert r3_by_address is not None and r3_by_pin is not None, "fill in both counts."
    assert r3_by_address == _addr, f"by-address count is off: expected {_addr}, got {r3_by_address}"
    assert r3_by_pin == _pin, f"by-pin count is off: expected {_pin}, got {r3_by_pin}"
    print(f"✅ Passed! Region III has {_addr} by address and {_pin} by pin.")
except AssertionError as e:
    print("❌ Not yet:", e)

### ❓ Question

Read the mismatch count printed earlier. How many institutions in the master have a pin that falls outside their recorded region?

In [ ]:
q_mismatch = None   # <- answer here (an integer)

In [ ]:
_mm = int((has_code["psgc_observed_region"] != has_code["expected_code"]).sum())
if q_mismatch == _mm:
    print(f"✅ Correct! {_mm} pins fall outside their recorded region.")
else:
    print(f"❌ Not yet: expected {_mm}, got {q_mismatch}.")

## Link qualifications and programs without a shared ID

The master has an institution ID. The assessment-center file does not. To put each accreditation on the map, every row needs the `tesda_inst_id` of its institution. The only thing the two files share is the **name** and the **address**, both typed inconsistently (`Inc.` against `Inc`, extra spaces, capitals).

Two ways to match on text:

- **Exact match on a normalized key.** Lower-case the name and address, strip everything except letters and digits, then compare. Fast and predictable. A single differing character means no match.
- **Fuzzy match on a similarity score.** Accept a pair above a similarity threshold. Recovers typos and abbreviations, at the risk of linking two different institutions that happen to look alike.

Exact normalization goes first because it is predictable and reviewable. Report the share it recovers, and leave the remainder for a fuzzy pass or a human.

In [ ]:
def normalize(text):
    return text.fillna("").str.lower().str.replace(r"[^a-z0-9]", "", regex=True)

# two spellings of one institution collapse to the same key
sample = pd.Series(["Datamex-College of Saint Adeline Inc.", "DATAMEX COLLEGE OF SAINT ADELINE, INC"])
print(normalize(sample).tolist())

# build name+address keys for the master's assessment-source rows
assess = master[master["source_record"] == "assessment"].copy()
assess["link_key"] = normalize(assess["name"]) + "|" + normalize(assess["address"])
key_to_id = assess.drop_duplicates("link_key")[["link_key", "tesda_inst_id"]]
print(f"{len(key_to_id):,} unique assessment keys")

## ✏️ Exercise - build the link key for the assessment file

Build the matching key for the assessment-center file the same way as for the master: the normalized `Assessment Center` name, a `|` separator, then the normalized `Address`. Store it in `ac["link_key"]`.

In [ ]:
### START CODE HERE ###   (about 1 line)
ac["link_key"] = None    # normalize(name) + "|" + normalize(address)
### END CODE HERE ###
ac["link_key"].head(3)

In [ ]:
try:
    ref_key = normalize(ac["Assessment Center"]) + "|" + normalize(ac["Address"])
    assert ac["link_key"].notna().all(), "link_key has blanks - fill in the code."
    assert (ac["link_key"] == ref_key).all(), "the key does not match the reference recipe."
    linked = ac.merge(key_to_id, on="link_key", how="left", validate="many_to_one")
    rate = linked["tesda_inst_id"].notna().mean()
    print(f"✅ Passed! AC rows linked to an institution: {rate:.1%} "
          f"({linked['tesda_inst_id'].isna().sum():,} of {len(linked):,} unlinked)")
except AssertionError as e:
    print("❌ Not yet:", e)

### Programs link the same way, and the real fix

The Compendium of Registered Programs links to the training-source rows of the master by the same normalized key. Both files drop their contact columns first.

In [ ]:
COMP_URL = BASE + "Compendium%20of%20Registered%20Programs%20as%20of%20June%202026.xlsx"
comp = pd.read_excel(COMP_URL, sheet_name="Compendium ao June 2026", dtype=str)
comp = comp.apply(lambda s: s.str.strip())
comp = comp.drop(columns=["TEL. NO.", "E-mail Address", "Name of Trainer/s"])

comp["link_key"] = normalize(comp["NAME OF INSTITUTION"]) + "|" + normalize(comp["ADDRESS"])
train = master[master["source_record"] == "training"].copy()
train["link_key"] = normalize(train["name"]) + "|" + normalize(train["address"])
key_to_id_train = train.drop_duplicates("link_key")[["link_key", "tesda_inst_id"]]

comp_linked = comp.merge(key_to_id_train, on="link_key", how="left", validate="many_to_one")
print(f"Compendium programs linked: {comp_linked['tesda_inst_id'].notna().mean():.1%} "
      f"of {len(comp_linked):,}")

The exact-key pass links roughly **89%** of accreditations and **93%** of programs. The rest fail because an address was typed differently in the two files (`Brgy.` against `Barangay`, a missing street number) or a name changed. A fuzzy pass could rescue some, at the risk of a wrong link.

The real fix is a shared **UIID**, a unique institution ID carried in every file. The Compendium workbook already has a `UIID` sheet. With that ID in the program sheet and in the assessment file, this whole section collapses to one reliable join, `ac.merge(master, on="uiid")`, with nothing lost. Report the link rate whenever a name-based link feeds an analysis.

## 🎛️ Play around - move the Philippine bounding box

The audit rests on one choice: the bounding box that defines inside the Philippines. Tighten the northern edge and watch clean northern pins turn into `out_of_ph`. Loosen it and watch borderline foreign pins slip back in. Change `lat_max` and compare the counts.

In [ ]:
def audit_counts(lat_max=21.5):
    global LAT_R, LON_R
    LAT_R, LON_R = (4.2, lat_max), (116.0, 127.0)
    q = [check_coords(a, b)[0] for a, b in zip(ac["Latitude"], ac["Longtitude"])]
    LAT_R, LON_R = (4.2, 21.5), (116.0, 127.0)   # restore the default box
    return pd.Series(q).value_counts()

lat_max = 21.5   # try 16.0 to push far-north pins out of bounds
print(f"lat_max={lat_max}")
print(audit_counts(lat_max))

In [ ]:
# Interactive in Colab / Jupyter / VS Code (static fallback elsewhere).
try:
    from ipywidgets import interact, FloatSlider
    interact(lambda lat_max=21.5: print(audit_counts(lat_max)),
             lat_max=FloatSlider(min=14.0, max=22.0, step=0.5, value=21.5))
except Exception:
    for lm in (16.0, 21.5):
        print(f"lat_max={lm} ->", audit_counts(lm).to_dict())

## References & further reading

- [pandas, user guide](https://pandas.pydata.org/docs/user_guide/)
- [pandas, merge and join](https://pandas.pydata.org/docs/user_guide/merging.html)
- [pandas, working with text data](https://pandas.pydata.org/docs/user_guide/text.html)
- [Philippine Standard Geographic Code (PSGC)](https://psa.gov.ph/classification/psgc)
- [Record linkage (overview)](https://en.wikipedia.org/wiki/Record_linkage)
- [Modifiable Areal Unit Problem (MAUP)](https://en.wikipedia.org/wiki/Modifiable_areal_unit_problem)

## Recap

- **One row is one accreditation** (one center can appear on many rows). 12,871 rows cover 2,727 distinct centers across 227 qualifications.
- Coordinates were typed by hand, so an audit comes first. Swapped latitude/longitude and dropped decimals are reversible and were repaired. Placeholders, out-of-country pins, and blanks keep no coordinate and carry a `coord_quality` flag.
- A numeric test alone accepts a pin in London. The flag is what keeps a wrong coordinate off the map.
- `psgc_observed_region` turns pin-against-address into a lookup. Counting **by address** and **by pin** exposes the gap one region at a time.
- With no shared ID, a normalized name-and-address key links about 89% of accreditations and 93% of programs. A shared UIID in every file would replace the whole linking step with one reliable join.

**For discussion**
1. Which checks here (coordinate status, pin against address, link rate) should run automatically whenever TESDA updates this list, and who should receive the flagged rows?
2. If the unlinked share of accreditations were concentrated in one region, how would that bias a coverage map of that region?
3. What is the smallest change to how centers are entered that would remove most of these fixes? (A PSGC code instead of free-text names, coordinates validated at entry, a UIID.)